
# 제품별 단일 프롬프트 생성 (옵션 B, v2: cluster/label/Description 주입)

각 (제품 × 페르소나) 조합에 대해 **클러스터 컨텍스트**(cluster/label/Description)를 포함한
싱글턴 프롬프트를 생성합니다.


In [1]:

# =============================
# 0) CONFIG
# =============================
from pathlib import Path

PERSONA_JSONL = Path("persona_attributes_weighted.jsonl")
PRODUCT_XLSX  = Path("product_info.xlsx")
OUT_JSONL     = Path("prompts_B.jsonl")
OUT_PREVIEW   = Path("prompts_B_preview.json")

LIMIT_PRODUCTS = None
LIMIT_PERSONAS = None

print("CONFIG loaded.")

CONFIG loaded.


In [2]:

# =============================
# 1) Load data
# =============================
import json, pandas as pd

# Personas
personas = []
with open(PERSONA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            personas.append(json.loads(line))

# Products
df_prod = pd.read_excel(PRODUCT_XLSX)

def get(d, k, default=""):
    return d[k] if k in d and pd.notna(d[k]) else default

products = []
for _, r in df_prod.iterrows():
    products.append({
        "product_id": get(r, "product_id", str(get(r, "id", ""))),
        "product_name": get(r, "product_name", get(r, "name", "")),
        "category": get(r, "category", ""),
        "features": get(r, "features", ""),
        "launch_ym": get(r, "launch_ym", ""),
        "price": get(r, "price", ""),
        "ad_model": get(r, "ad_model", ""),
    })

if LIMIT_PRODUCTS: products = products[:LIMIT_PRODUCTS]
if LIMIT_PERSONAS: personas = personas[:LIMIT_PERSONAS]

len(personas), len(products)

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [ ]:

# =============================
# 2) Helpers
# =============================
from typing import Dict, Any

def format_attributes_for_prompt(attrs: Dict[str, Any]) -> str:
    lines = []
    for k, vw in attrs.items():
        v = vw.get("value", None)
        w = vw.get("weight", 0.0)
        v_str = "None" if v is None else str(v)
        lines.append(f"- {k}: {v_str} (w={w:.3f})")
    return "\n".join(lines[:60])

def build_single_prompt(product: Dict[str, Any], persona: Dict[str, Any]) -> str:
    meta = persona.get("meta", {}) or {}
    cluster = meta.get("cluster", "")
    label = meta.get("label", "")
    desc = meta.get("desc", "")
    cluster_block = f"""
[클러스터 컨텍스트]
- cluster: {cluster}
- label: {label}
- desc: {desc}
""".strip() if (cluster or label or desc) else "[클러스터 컨텍스트]\n- N/A"

    return f"""
[역할]
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.
아래의 "제품 정보"와 "페르소나"를 바탕으로,
이 페르소나가 해당 제품의 구매자로서 성립하는 **싱글 턴** 페르소나 JSON을 생성하세요.

[제품 정보]
- product_id: {product.get('product_id','')}
- 제품명: {product.get('product_name','')}
- 출시월(YYYY-MM): {product.get('launch_ym','')}
- 카테고리: {product.get('category','')}
- 주요 특징: {product.get('features','')}
- 기준 가격대(원): {product.get('price','')}
- 광고모델: {product.get('ad_model','')}

[페르소나]
- id: {persona.get('persona_key','')}
- 속성(가중치 합=1):
{format_attributes_for_prompt(persona.get('attributes', {}))}

{cluster_block}

[규칙]
- '클러스터 컨텍스트'는 배경 지침으로만 사용합니다. 속성 가중치(합=1)와 충돌 시 '속성 가중치'를 우선합니다.
- 2024-07 ~ 2025-06 월별로 구매확률(prob 0~1)과 예상수량(qty 정수)을 제시합니다.
- 추석/설, 광고, 계절성을 반영합니다.
- **반드시 아래 JSON 스키마를 출력**하고, 불필요한 설명 문장은 출력하지 마세요.

[출력 스키마(JSON)]
{{
  "persona_id": "p_{product.get('product_id','')}_{persona.get('persona_key','')}",
  "product_id": "{product.get('product_id','')}",
  "segment_ref": "{persona.get('persona_key','')}",
  "attributes": { "{'{'}속성명{'}'}": { "{'{'}value{'}'}": "<값>", "{'{'}weight{'}'}": <0~1> }, "...": "..." },
  "purchase_pattern": {{
    "avg_purchase_prob": <0~1>,
    "avg_purchase_qty": <int>,
    "seasonality": {{"추석": "+x%", "설": "+y%"}},
    "promotion_effect": "광고모델 노출 시 +z%"
  }},
  "forecast_12mo": {{
    "2024-07": {{"prob": <0~1>, "qty": <int>}},
    "...": {{}}, 
    "2025-06": {{"prob": <0~1>, "qty": <int>}}
  }}
}}
""".strip()

In [ ]:

# =============================
# 3) Build & save
# =============================
import json

records = []
for prod in products:
    for persona in personas:
        rec = {
            "product": prod,
            "persona": {"persona_key": persona.get("persona_key")},
            "prompt": build_single_prompt(prod, persona)
        }
        records.append(rec)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

from pathlib import Path
Path(OUT_PREVIEW).write_text(json.dumps(records[:3], ensure_ascii=False, indent=2), encoding="utf-8")
len(records), OUT_JSONL, OUT_PREVIEW